# Planetary Rover GRPO Training (Colab)

This notebook runs the same GRPO pipeline as `train.py` for the OpenEnv hackathon submission.

Pipeline:
- Llama 3.2 1B + Unsloth 4-bit NF4 + LoRA
- TRL `GRPOTrainer`
- Two-tier rewards: format gatekeeper + environment reward from FastAPI physics server

## 1) Prepare Runtime

In Colab, place this repo at `/content/Scaler_meta_hugging` (clone or upload) before running the next cell.

In [ ]:
import os
import pathlib

if pathlib.Path('/content').exists():
    repo_path = pathlib.Path('/content/Scaler_meta_hugging')
    if not repo_path.exists():
        raise FileNotFoundError('Expected repo at /content/Scaler_meta_hugging. Clone/upload it first.')
    os.chdir(repo_path)

print('Working directory:', os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import requests
import subprocess
import sys
import time

server_proc = subprocess.Popen([
    sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '7860'
])

deadline = time.time() + 30
server_ready = False
while time.time() < deadline:
    try:
        res = requests.get('http://127.0.0.1:7860/tasks', timeout=2)
        if res.ok:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    raise RuntimeError('Server failed to start within 30 seconds.')

print('FastAPI environment server is live on :7860')

## 2) Run GRPO Training

This cell imports `train.py`, sets runtime knobs, and starts training.
Look for `METRICS {'loss': ..., 'reward': ...}` in output for your submission screenshot.

In [ ]:
import os
import train as rover_train

os.environ['ROVER_SERVER_URL'] = 'http://127.0.0.1:7860'
os.environ['ROVER_NUM_GENERATIONS'] = '8'

rover_train.SERVER_URL = os.environ['ROVER_SERVER_URL']
rover_train.NUM_GENERATIONS = int(os.environ['ROVER_NUM_GENERATIONS'])

rover_train.main()

In [ ]:
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('Stopped FastAPI environment server.')
else:
    print('No running server process found.')